# VM End-To-End Setup

Notebook nay dung de setup lai mot GPU VM tu dau va chay full end-to-end tren VM:

```text
Expo app -> Cloudflare backend tunnel -> FastAPI backend on VM
  -> local bbox cleanup
  -> Hunyuan worker on same VM
  -> GLB result
```

Khong dung Gemini/Nano Banana. Khong dung TripoSR. Backend tren VM se goi worker noi bo qua `http://127.0.0.1:8010`.

## 0. Config

Sua `REPO_REF` neu ban da merge branch nay vao main.

In [ ]:
REPO_URL = "https://github.com/TangDien02/AI_3D_Reconstruction_Systerm.git"
REPO_REF = "codex/hunyuan-shape-then-paint"
WORK_DIR = "$HOME/work"
REPO_DIR = "$HOME/work/AI_3D_Reconstruction_Systerm"
HUNYUAN_DIR = "$HOME/work/Hunyuan3D-2"
print(REPO_URL, REPO_REF)

## 1. Check VM GPU

In [ ]:
!nvidia-smi
!python3 --version
!free -h
!df -h | head -20

## 2. Install base tools and cloudflared

In [ ]:
%%bash
set -euo pipefail
sudo apt-get update
sudo apt-get install -y --no-install-recommends git curl wget tmux build-essential python3-pip python3-venv ninja-build ca-certificates
if ! command -v cloudflared >/dev/null 2>&1; then
  curl -L -o /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
  sudo dpkg -i /tmp/cloudflared.deb || sudo apt-get install -f -y
fi
cloudflared --version

## 3. Clone repo branch

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/TangDien02/AI_3D_Reconstruction_Systerm.git"
REPO_REF="codex/hunyuan-shape-then-paint"
WORK_DIR="$HOME/work"
REPO_DIR="$WORK_DIR/AI_3D_Reconstruction_Systerm"
mkdir -p "$WORK_DIR"
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone --branch "$REPO_REF" "$REPO_URL" "$REPO_DIR"
else
  git -C "$REPO_DIR" fetch origin "$REPO_REF"
  git -C "$REPO_DIR" checkout "$REPO_REF"
  git -C "$REPO_DIR" pull --ff-only
fi
git -C "$REPO_DIR" status --short
git -C "$REPO_DIR" rev-parse --short HEAD

## 4. Setup Hunyuan worker service

Cell nay cai Hunyuan3D-2 vao `$HOME/work/venv`, copy worker FastAPI, va tao service `hunyuan-worker` port `8010`.

In [ ]:
%%bash
set -euo pipefail
cd "$HOME/work/AI_3D_Reconstruction_Systerm"
export REPO_REF="codex/hunyuan-shape-then-paint"
bash scripts/gcp_hunyuan_worker_bootstrap.sh

## 5. Verify worker

In [ ]:
!curl -s http://127.0.0.1:8010/health
!sudo systemctl status hunyuan-worker --no-pager | head -40

## 6. Setup FastAPI backend on the same VM

Backend VM se goi Hunyuan worker noi bo qua `http://127.0.0.1:8010`, nen khong can worker tunnel rieng.

In [ ]:
%%bash
set -euo pipefail
cd "$HOME/work/AI_3D_Reconstruction_Systerm"
export HUNYUAN_REMOTE_URL="http://127.0.0.1:8010"
bash scripts/gcp_backend_vm_bootstrap.sh

## 7. Verify backend

In [ ]:
!curl -s http://127.0.0.1:8000/health
!sudo systemctl status ai-3d-backend --no-pager | head -50

## 8. Start Cloudflare tunnel for backend

Expo se goi URL nay. Copy URL `https://....trycloudflare.com` tu output cell.

In [ ]:
%%bash
set -euo pipefail
cd "$HOME/work/AI_3D_Reconstruction_Systerm"
SESSION=backend-tunnel TARGET_URL=http://127.0.0.1:8000 bash deploy/scripts/start_tunnel_tmux.sh
sleep 8
tmux capture-pane -t backend-tunnel -p -S -120 | tee /tmp/backend_tunnel.log
grep -o 'https://[^ ]*trycloudflare.com' /tmp/backend_tunnel.log | tail -1 || true

## 9. Smoke test backend tunnel

Paste backend tunnel URL vao bien duoi roi chay cell.

In [ ]:
BACKEND_TUNNEL_URL = "https://YOUR_BACKEND_TUNNEL.trycloudflare.com"  # replace this
print(BACKEND_TUNNEL_URL)

In [ ]:
import json, urllib.request
with urllib.request.urlopen(BACKEND_TUNNEL_URL.rstrip('/') + '/health', timeout=30) as r:
    print(json.dumps(json.load(r), indent=2)[:2000])

## 10. Local clean smoke test

Test preprocess endpoint truoc khi gui job Hunyuan.

In [ ]:
%%bash
set -euo pipefail
curl -s -X POST "http://127.0.0.1:8000/preprocess/clean-image" \
  -F "image=@$HOME/work/AI_3D_Reconstruction_Systerm/project/samples/chair_demo.png" \
  -F "bbox_x=10" \
  -F "bbox_y=10" \
  -F "bbox_width=400" \
  -F "bbox_height=400" \
  -F "job_id=vm-clean-smoke" | python3 -m json.tool | head -80

## 11. End-to-end reconstruct smoke test

Cell nay tao job Hunyuan that, co the mat vai phut. Neu worker dang busy, doi job truoc xong.

In [ ]:
%%bash
set -euo pipefail
RESP=$(curl -s -X POST "http://127.0.0.1:8000/reconstruct-bbox" \
  -F "image=@$HOME/work/AI_3D_Reconstruction_Systerm/project/samples/chair_demo.png" \
  -F "bbox_x=10" \
  -F "bbox_y=10" \
  -F "bbox_width=400" \
  -F "bbox_height=400")
echo "$RESP" | python3 -m json.tool
JOB_ID=$(printf '%s' "$RESP" | python3 -c "import sys,json; print(json.load(sys.stdin)['job_id'])")
echo "JOB_ID=$JOB_ID"
for i in $(seq 1 120); do
  STATUS=$(curl -s "http://127.0.0.1:8000/reconstruction-jobs/$JOB_ID")
  echo "$STATUS" | python3 -m json.tool | head -40
  echo "$STATUS" | grep -q '"status": "done"' && break
  echo "$STATUS" | grep -q '"status": "failed"' && exit 1
  sleep 5
done

## 12. Expo command on laptop

Tren Windows/laptop, restart Expo voi backend tunnel URL lay o buoc 8:

```powershell
cd mobile
$env:EXPO_PUBLIC_API_BASE_URL="https://YOUR_BACKEND_TUNNEL.trycloudflare.com"
npm start
```

Neu dung backend local Windows thi van co the dung:

```powershell
$env:EXPO_PUBLIC_API_BASE_URL="http://192.168.1.5:8000"
npm start
```

## 13. Logs

In [ ]:
# Worker logs
!sudo journalctl -u hunyuan-worker -n 120 --no-pager

In [ ]:
# Backend logs
!sudo journalctl -u ai-3d-backend -n 120 --no-pager

In [ ]:
# Backend tunnel logs
!tmux capture-pane -t backend-tunnel -p -S -120

## 14. Restart commands

```bash
sudo systemctl restart hunyuan-worker
sudo systemctl restart ai-3d-backend
tmux kill-session -t backend-tunnel
SESSION=backend-tunnel TARGET_URL=http://127.0.0.1:8000 bash ~/work/AI_3D_Reconstruction_Systerm/deploy/scripts/start_tunnel_tmux.sh
```